Single allele groupped structue models.

In [11]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [12]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/alleles_with_groups_single_allele_balanced_data/" #!CHANGE for bwv!

In [13]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*01:01',
 'HLA-A*03:01',
 'HLA-A*02:01',
 'HLA-B*07:02',
 'HLA-B*27:05',
 'HLA-DRB1*01:01']

## 2.0 Merge alleles together

In [14]:
# combination_size = len(ALLELES)
ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
ALLELES = sorted(['HLA-DRB1*01:01'])


combination_size = 1
TOTAL_SPLITS = 5

In [15]:
import itertools
from collections import defaultdict
per_allele_per_kfold_per_length_binders_train_new = dict()

per_allele_per_kfold_per_length_binders_test_new = dict()

per_kfold_per_length_per_allele_counts_old_train = dict()
per_kfold_per_length_per_allele_counts_old_test = dict()





OLD_ALLELES = ALLELES
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)

# initialize count dicts
for allele_name in OLD_ALLELES:
    for split_num in range(TOTAL_SPLITS):
        per_kfold_per_length_per_allele_counts_old_train[split_num] = defaultdict(int)
        per_kfold_per_length_per_allele_counts_old_test[split_num] = defaultdict(int)
        
    



for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()


    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        # #calculate allele_weights here
        # # allele weight is   len(allele data)/ total_data_length
        # for old_allele in combination:
        #     lengths_list = {
        #         length: len(length_binders)
        #         for length, length_binders
        #         in per_allele_per_kfold_per_length_binders_train[old_allele][split_num].values()
        #     }


        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if target_length not in per_kfold_per_length_per_allele_counts_old_train[split_num]:
                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length] = dict()
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length] = dict()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]

                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length][old_allele] = len(single_allele_data_array_train)
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)
                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length][old_allele] = len(single_allele_data_array_test)
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            if split_num == 0:
                print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)


per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-DRB1*01:01
Length 8: HLA-DRB1*01:01 - 39 
Length 9: HLA-DRB1*01:01 - 113 
Length 10: HLA-DRB1*01:01 - 123 
Length 11: HLA-DRB1*01:01 - 175 
Length 12: HLA-DRB1*01:01 - 368 
Length 13: HLA-DRB1*01:01 - 826 
Length 14: HLA-DRB1*01:01 - 1184 
Length 15: HLA-DRB1*01:01 - 6886 
Length 16: HLA-DRB1*01:01 - 1413 
Length 17: HLA-DRB1*01:01 - 984 
Length 18: HLA-DRB1*01:01 - 708 
Length 19: HLA-DRB1*01:01 - 316 
Length 20: HLA-DRB1*01:01 - 505 
Length 21: HLA-DRB1*01:01 - 177 
Length 22: HLA-DRB1*01:01 - 80 
Length 23: HLA-DRB1*01:01 - 57 
Length 24: HLA-DRB1*01:01 - 67 
Length 25: HLA-DRB1*01:01 - 50 
Length 26: HLA-DRB1*01:01 - 15 
Length 27: HLA-DRB1*01:01 - 12 
Length 28: HLA-DRB1*01:01 - 8 
Length 30: HLA-DRB1*01:01 - 7 


## Calculate weights for data

In [16]:

def extract_count_df(per_kfold_per_length_per_allele_counts):
    result_dict = {}
    for kfold,  per_length_per_allele_counts in per_kfold_per_length_per_allele_counts.items():
        for length, per_allele_counts in per_length_per_allele_counts.items():
            for allele_name , count in  per_allele_counts.items():
                result_dict[(allele_name, kfold, length)] =  {'cnt': count }
    count_df = pd.DataFrame.from_dict(result_dict, orient="index")
    count_df.index.names = ['allele', 'split', 'length']
    # count_df = count_df.groupby(['allele', 'length'])['cnt'].mean()
    return count_df



count_df_train = extract_count_df(per_kfold_per_length_per_allele_counts_old_train)
count_df_test = extract_count_df(per_kfold_per_length_per_allele_counts_old_test)


In [17]:
count_df_train.index.levels

FrozenList([['HLA-DRB1*01:01'], [0, 1, 2, 3, 4], [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30]])

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [18]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model

TARGET_LENGTHS= [13, 14, 15, 16, 17, 18]
NUM_RUNS = 10
TOTAL_SPLITS = 5
USE_WEIGHTS = True

### Define weights for data

In [19]:
per_allele_per_kfold_per_length_weights = dict()


for combined_allele_name in ALLELES:
    per_allele_per_kfold_per_length_weights[combined_allele_name] = dict()
    old_alleles = combined_allele_name.split('+')
    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_weights[combined_allele_name][split_num] = dict()
        sub_count_df_train = count_df_train.loc[(count_df_train.index.get_level_values('split') == 0) &
                                                (count_df_train.index.get_level_values('allele').isin(old_alleles)) &
                                                (count_df_train.index.get_level_values('length').isin(TARGET_LENGTHS))]
        total_data_length = sub_count_df_train['cnt'].sum()
        print(f"Total data length for split {split_num} is {total_data_length}")
        for target_length in TARGET_LENGTHS:

            current_weights_list = list()
            weights_dict = dict()
            for old_allele_name in old_alleles:
                old_allele_cnt = count_df_train.loc[(old_allele_name, split_num, target_length)]['cnt']
                current_weights_list.extend([total_data_length/old_allele_cnt if USE_WEIGHTS else 1 for i in range(old_allele_cnt)])
                weights_dict[old_allele_name] = total_data_length/old_allele_cnt if USE_WEIGHTS else 1
            per_allele_per_kfold_per_length_weights[combined_allele_name][split_num][target_length] = current_weights_list
            print(f"Weights for lentgh {target_length} are {weights_dict}")


Total data length for split 0 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01': 10.13597972972973}
Weights for lentgh 15 are {'HLA-DRB1*01:01': 1.7428115015974441}
Weights for lentgh 16 are {'HLA-DRB1*01:01': 8.493276716206653}
Weights for lentgh 17 are {'HLA-DRB1*01:01': 12.196138211382113}
Weights for lentgh 18 are {'HLA-DRB1*01:01': 16.95056497175141}
Total data length for split 1 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01': 10.127426160337553}
Weights for lentgh 15 are {'HLA-DRB1*01:01': 1.7428115015974441}
Weights for lentgh 16 are {'HLA-DRB1*01:01': 8.493276716206653}
Weights for lentgh 17 are {'HLA-DRB1*01:01': 12.196138211382113}
Weights for lentgh 18 are {'HLA-DRB1*01:01': 16.95056497175141}
Total data length for split 2 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01'

## 2.2 Train
### 2.2.1 Define treshold

In [20]:
stop_threshold = 0
cutoff = 0.0005
per_symbol_portion = -50 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

84.57592592592592

### 2.2.2 Prepare models

In [21]:
# Training params
algorithm = 'baum-welch'

params = dict()


params['st'] = 49
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 5
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 1
params['transition_pseudocount'] = 0
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0.25
params['distribution_inertia'] = 0
params['groups'] = 7

assert params['st'] % params['groups'] == 0

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))


In [22]:
model = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

grouped_model : grouped_model-start summed to 7.0, normalized to 1.0
grouped_model : s000 summed to 31.0, normalized to 1.0
grouped_model : s001 summed to 31.0, normalized to 1.0
grouped_model : s002 summed to 31.0, normalized to 1.0
grouped_model : s003 summed to 31.0, normalized to 1.0
grouped_model : s004 summed to 31.0, normalized to 1.0
grouped_model : s005 summed to 31.0, normalized to 1.0
grouped_model : s006 summed to 31.0, normalized to 1.0
grouped_model : s007 summed to 31.0, normalized to 1.0
grouped_model : s008 summed to 31.0, normalized to 1.0
grouped_model : s009 summed to 31.0, normalized to 1.0
grouped_model : s010 summed to 31.0, normalized to 1.0
grouped_model : s011 summed to 31.0, normalized to 1.0
grouped_model : s012 summed to 31.0, normalized to 1.0
grouped_model : s013 summed to 31.0, normalized to 1.0
grouped_model : s014 summed to 31.0, normalized to 1.0
grouped_model : s015 summed to 31.0, normalized to 1.0
grouped_model : s016 summed to 31.0, normalized to 

In [23]:
model.dense_transition_matrix()

array([[0.        , 0.12903226, 0.12903226, ..., 0.        , 0.        ,
        0.        ],
       [0.12903226, 0.        , 0.12903226, ..., 0.        , 0.        ,
        0.        ],
       [0.12903226, 0.12903226, 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.14285714],
       [0.14285714, 0.14285714, 0.14285714, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [ ]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [25]:
ALLELES

['HLA-DRB1*01:01']

In [32]:
%%time


per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]


        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'], use_test=False)
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


run 0
split 0
train size 12001
Shuffling for fairness 
Simple training: Train Length 12001, init_batch_size None
[1] Improvement: 29126.866846843623	Time (s): 11.24
[2] Improvement: 2181.729855627753	Time (s): 11.17
[3] Improvement: 1340.8390700728633	Time (s): 11.41
[4] Improvement: 1130.3093641414307	Time (s): 11.23
[5] Improvement: 1033.9578782166354	Time (s): 11.42


KeyboardInterrupt: 

In [27]:
from hmm_visualization_methods import *

In [28]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

Wall time: 0 ns


In [29]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

Wall time: 0 ns


In [30]:
from hmm_visualization_methods import *
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=2)

In [31]:
TARGET_PATH_TO_RESULTS

'experiment_results/alleles_with_groups_single_allele_balanced_data/'

## Aux viterbi

In [49]:
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [50]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]
        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm='viterbi',
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount= 1,  #!CHANGE for bwv!
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia= params['distribution_inertia'],
                                        lr_decay=params['lr_decay']
                                        )
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 12001
Shuffling for fairness 
Simple training: Train Length 12001, init_batch_size None
[1] Improvement: -2737.7927925364347	Time (s): 4.856
[2] Improvement: -1021.6540945441229	Time (s): 4.698
[3] Improvement: -220.54597513540648	Time (s): 4.774
[4] Improvement: -46.667490428546444	Time (s): 4.758
[5] Improvement: 37.480640860972926	Time (s): 4.706
Total Training Improvement: -3989.1797117835376
Total Training Time (s): 28.6110
split 0
train size 12001
Shuffling for fairness 
Simple training: Train Length 12001, init_batch_size None
[1] Improvement: -3137.2437254369725	Time (s): 6.606
[2] Improvement: -968.4346556843957	Time (s): 5.508
[3] Improvement: -178.12960512621794	Time (s): 5.179
[4] Improvement: -4.8563815174857154	Time (s): 4.857
[5] Improvement: 43.35328758799005	Time (s): 4.664
Total Training Improvement: -4245.311080177082
Total Training Time (s): 31.6600
split 0
train size 12001
Shuffling for fairness 
Simple training: Train Length 12001, init_batch_si

In [51]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
Wall time: 17.4 s


In [52]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
Wall time: 4min 14s


In [53]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (2_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (3_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (4_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (5_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (6_run-13-14-15-16-17-18-35-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi) (Delete zero edges)
Conver